In [1]:
%pip install dash
%pip install pandas
%pip install statsmodels
%pip install matplotlib


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\jakob\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\jakob\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\jakob\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\jakob\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [14]:
from dash import Dash, html, dcc, Input, Output, callback, Output, Input
import plotly.express as px
import pandas as pd
import json
import statistics 
import plotly.graph_objects as go

from Q1 import df1
from Q4 import slope_df
from Q5 import outcome_pct_q5
from Q6 import df6
from Q7 import df7
#from Q8 import df8


pd.options.plotting.backend = 'plotly'

### dataframes, plot computation, figures ###
with open("./.data_Q2.txt", "r") as file:
    data_Q2 = json.loads(file.read())

with open("./.data_Q3.txt", "r") as file:
    data_Q3 = json.loads(file.read())


def filter_prec(data, threshold):
    teams_lt = []
    goals_lt = []
    teams_gt = []
    goals_gt =[]
    df1 = pd.DataFrame()
    df2 = pd.DataFrame()
    for team, matches in data.items():
        for match in list(zip(*matches)):
            if match[1] > threshold:
                teams_gt.append(team)
                goals_gt.append(match[0])
            else:
                teams_lt.append(team)
                goals_lt.append(match[0])
    df1["Teams"] = teams_lt
    df1["Goals"] = goals_lt
    df2["Teams"] = teams_gt
    df2["Goals"] = goals_gt
    return px.box(df1, x="Teams", y="Goals"), px.box(df2,x="Teams", y="Goals")
    

fig_Q2_0, fig_Q2_1 = filter_prec(data_Q2, 1.5)

fig_Q3_0 = px.scatter(data_Q3)
df3 = pd.DataFrame()
avgs = []
precs = []
for i in range(0, 350):
    matches = list(filter(lambda x: i/10 <= x[1] < i/10+0.1, list(zip(data_Q3["Goals"], data_Q3["Precipitation"]))))
    if len(matches) > 5:
        avgs.append(statistics.mean(x for (x, _) in matches))
        precs.append(i/10)

df3["Average goals per match"] = avgs
df3["Precipitation group"] = precs
fig_Q3_1 = px.scatter(df3, x="Precipitation group",y="Average goals per match", trendline="ols")

df1 = df1()
df6 = df6()
df7 = df7()
#df8 = df8()

fig1 = px.bar(df1, x = 'Minute', y = 'Percentage', color = 'Team', barmode='group')

fig_q4 = go.Figure()

for i, row in slope_df.iterrows():
    #line between the 2 ranks(wincount and comeback %)
    fig_q4.add_trace(go.Scatter(
        x=[0, 1],
        y=[row["rank_wins"], row["rank_rate"]],
        mode='lines+markers',
        line=dict(color=row["color"], width=2),
        marker=dict(size=7, color=row["color"]),
        showlegend=False,
        hovertemplate=f"{row['team']}<br>Comebackwins Rank: #{row['rank_wins']}<br>Comebackrate Rank: #{row['rank_rate']}<extra></extra>"
    ))
    #teamnames left
    fig_q4.add_annotation(x=0, y=row["rank_wins"], text=row["team"],
                          xanchor="right", showarrow=False, font=dict(size=9))
    #teamnames right
    fig_q4.add_annotation(x=1, y=row["rank_rate"], text=row["team"],
                          xanchor="left", showarrow=False, font=dict(size=9))

fig_q4.update_layout(
    xaxis=dict(
        tickvals=[0, 1],
        ticktext=["Rank by Comeback Wins", "Rank by Comeback Rate"],
        range=[-0.5, 1.5]
    ),
    yaxis=dict(autorange="reversed", title="Rank"),
    height=700,
)

### website initialization, layout ###

# Initialize the app
app = Dash()

# App layout
app.layout = [
    html.Div([
        html.H1('[Title]', style = {'textAlign': 'center'}),
        html.H2('[Subtitle]', style = {'textAlign': 'center'}),
        html.Div([
            html.H3('Question 1: Inspecting 15 minute intervalls, when during the last 15 years (seasons 10/11 to 24/25) were the most goals scored?'),
            html.Div(dcc.Dropdown(df1['Team'][::7], value = ['Bundesliga'], multi = True, id = 'Q1TeamDropdown')),
            html.Div(dcc.RadioItems(options = ['Goals', 'Percentage'], value = 'Goals', id = 'Q1Radio')),
            html.Div(dcc.Graph(id = 'Q1Barchart', figure = fig1))   
        ], id = 'Q1Div'),
        html.Div([
            html.H3('[Question 2]'),
            html.P("Select the amount of precipitation to be used as threshold:"),
            dcc.Slider(
                id='Q2_rain_filter',
                min=0,
                max=35,
                step=0.1,
                value=1.5,
            ),
            html.P("Select any number of teams for the charts:"),
            dcc.Dropdown(list(data_Q2.keys()), list(data_Q2.keys()), True, True, True, id="Q2_teams_filter"),
            html.Div(children=dcc.Graph(id= "Q2_0", figure=fig_Q2_0)),
            html.Div(children=dcc.Graph(id= "Q2_1", figure=fig_Q2_1)),
        ], id = 'Q2Div'),
        html.Div([
            html.H3('[Question 3]'),
            html.Div(children=dcc.Graph(id= "Q3_0", figure=fig_Q3_0)),
            html.Div(children=dcc.Graph(id= "Q3_1", figure=fig_Q3_1)),
        ], id = 'Q3Div'),
        html.Div([
        html.H3('Question 4: Which teams are the best at staging comebacks after trailing at half time?'),
        dcc.Graph(id='q4_static', figure=fig_q4),
        dcc.RadioItems(
            id='q4_radio',
            options=[
                    {'label': '  Comeback Wins', 'value': 'comeback_wins'},
                    {'label': '  Comeback Rate', 'value': 'comeback_rate_%'},
            ],
            value='comeback_wins',
            inline=True,
        ),
        dcc.Graph(id='q4_sort'),
    ], id='Q4Div'),
        html.Div([
            html.H3('Question 5: How does the timing of the first conceded lead-giving goal affect the match outcome?'),
            dcc.RadioItems(
            id='q5_outcome_filter',
            options=[
                {'label': '  Win',  'value': 'win'},
                {'label': '  Draw', 'value': 'draw'},
                {'label': '  Loss', 'value': 'loss'},
            ],
            value='loss',
            inline=True,
            style={'marginBottom': '10px'}
            ),
            dcc.Graph(id='q5_outcome_focus'),
            html.Div() # Some graph, chart, plot, etc.
        ], id = 'Q5Div'),
        html.Div([
            html.H3('Question 6: Which teams scored goals at home turf most often?'),
            html.Div(dcc.Slider(2010, 2024, 1, value=2010, id='component6')),
            html.Div(dcc.Graph(id = 'graph6')), 
        ], id = 'Q6Div'),
        html.Div([
            html.H3('Question 7: During the last 15 years (seasons 09/10 to 24/25), \
                when did each team score goals most often in their opponent`s city?'),
            dcc.Dropdown(
                options = ['1. FSV Mainz 05', 'TSG Hoffenheim', 'Bayer 04 Leverkusen', \
                    'FC Bayern München', 'Borussia Dortmund', 'Borussia Mönchengladbach', \
                    'VfL Wolfsburg'], 
                value = ['FC Bayern München'], 
                multi = True,  
                id = 'component7'),
            html.Div(dcc.Graph(id = 'graph7'))
        ], id = 'Q7Div')
        
        #''',
        #html.Div([
        #    html.H3('Question 8: How does the venue capacity influence the win rate of the \
        #        away team for the seasons 2022-2024?'),
        #    dcc.RangeSlider(10000, 100000, 1000, value=[10000, 100000], id = 'component8_1'),
        #    dcc.Dropdown(
        #        options = ['1. FC Heidenheim 1846', '1. FC Köln',	'1. FC Union Berlin', '1. FSV Mainz 05', \
        #            'Bayer 04 Leverkusen', 'Borussia Dortmund', 'Borussia Mönchengladbach', \
        #            'Eintracht Frankfurt', 'FC Augsburg', 'FC Bayern München', 'FC Schalke 04', 'FC St. Pauli', \
        #            'Hertha BSC', 'RB Leipzig', 'SC Freiburg', 'SV Darmstadt 98', 'SV Werder Bremen', \
        #            'TSG Hoffenheim', 'VfB Stuttgart', 'VfL Bochum', 'VfL Wolfsburg'], 
        #        value = ['FC Bayern München'],
        #        multi = True, 
        #        id = 'component8_2'),
        #        html.Div(dcc.Graph(id = 'graph8', figure = px.scatter(df8))) 
        #], id = 'Q8Div'),
        #'''
    ])
]

### Callbacks ###

# Q1 Barchart 
@callback(
    Output('Q1Barchart', 'figure'),
    Input('Q1Radio', 'value'),
    Input('Q1TeamDropdown','value')
)
def update_Q1(ratio, selected_teams):
    df_temp = pd.DataFrame({})
    for team in selected_teams:
        rows = df1.loc[df1['Team'] == team]
        df_temp = pd.concat([df_temp, rows], ignore_index= True)
    fig1 = px.bar(df_temp, x = 'Minute', y = ratio, color = 'Team', barmode='group')
    return fig1

@callback(
    Output(component_id="Q2_0", component_property="figure"),
    Output(component_id="Q2_1", component_property="figure"),
    Input(component_id="Q2_rain_filter", component_property="value"),
    Input(component_id="Q2_teams_filter", component_property="value")
)
def update_plots_Q2(threshold, selection):
    data = dict(list(filter(lambda x: x[0] in selection, data_Q2.items())))
    return filter_prec(data, threshold)

@callback(
    Output('q5_outcome_focus', 'figure'),
    Input('q5_outcome_filter', 'value')
)

def update_q5_outcome(selected_outcome):
    color_map = {'win': 'green', 'draw': 'grey', 'loss': 'red'}
    x = outcome_pct_q5.index.astype(str).tolist()
    y = outcome_pct_q5[selected_outcome]
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=x,
        y=y,
        marker_color=color_map[selected_outcome],
        name=selected_outcome.capitalize()
    ))
    fig.update_layout(
        title=f'Share of "{selected_outcome.capitalize()}" by Time Bins Bundesliga 2009-2024',
        xaxis_title='Minute of First Lead-Giving Goal Conceded',
        yaxis_title='Share of Matches (%)',
        yaxis=dict(range=[0, 105]),
    )
    return fig

@callback(
    Output('q4_sort', 'figure'),
    Input('q4_radio', 'value')
)

def update_q4(sort_by):
    filtered = slope_df.sort_values(sort_by, ascending=False).head(15)
    fig = px.scatter(filtered, x=sort_by, y='team', color=sort_by,
                     color_continuous_scale=px.colors.sequential.Greens[3:],
                     title=f'Top 15 Teams sorted by {sort_by}')
    fig.update_traces(marker=dict(size=12))
    fig.update_layout(yaxis=dict(autorange='reversed'))
    return fig

@callback(
    Output(component_id = 'graph6', component_property = 'figure'),
    Input(component_id = 'component6', component_property = 'value')
)
def upgrade_graph_6(value_chosen):
    return px.bar(df6, y = value_chosen)

@callback(
    Output(component_id = 'graph7', component_property = 'figure'),
    Input(component_id = 'component7', component_property = 'value')
)
def upgrade_graph_7(value_chosen):
    return px.line(df7, y = value_chosen)

'''
@callback(
    Output(component_id = 'graph8', component_property = 'figure'),
    Input(component_id = 'component8_1', component_property = 'value'),
    Input(component_id = 'component8_2', component_property = 'value')
)
def upgrade_graph_8(slider, value_chosen): 
    df_temp = pd.DataFrame({})
    rows = df8.loc[(df8.index >= slider[0]) & (df8.index <= slider[1])]
    df_temp = pd.concat([df_temp, rows])
    return px.scatter(df_temp, y = value_chosen)
'''


### run the app ###
# Run the app
if __name__ == '__main__':
    app.run(debug=True, port = 8080)
